In [ ]:
import os
import subprocess
import sys
import importlib.metadata

gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
).decode().strip()

print("GPU détecté :", gpu_info)

torch_version = None
try:
    torch_version = importlib.metadata.version("torch")
except:
    pass

if "P100" in gpu_info:
    if torch_version != "2.6.0":
        print("Installation PyTorch compatible P100...")

        subprocess.run([
            sys.executable, "-m", "pip", "install",
            "torch==2.6.0",
            "torchvision==0.21.0",
            "torchaudio==2.6.0",
            "--index-url",
            "https://download.pytorch.org/whl/cu126"
        ])

    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "ultralytics"
    ])

else:
    print("GPU moderne détecté, installation légère.")

    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "ultralytics"
    ])

In [ ]:
import os
import cv2
import yaml
import shutil
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
import mlflow
from kaggle_secrets import UserSecretsClient

In [ ]:
user_secrets = UserSecretsClient()

MLFLOW_USER = "mlflowsuperuser"
MLFLOW_PASS = "wfu:1H6B_pFLe',2"
MLFLOW_HOST = "70.50.155.244"

TRACKING_URI = f"http://{MLFLOW_USER}:{MLFLOW_PASS}@{MLFLOW_HOST}:5000"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("foodseg-dataset-scaling")

In [ ]:
ROOT = "/kaggle/input/foodseg103"

FULL_DATASET = "/kaggle/working/full_yolo_dataset"
EXPERIMENTS = [100, 300, 500, 700, 1000]
RUNS_PER_SIZE = 3

MODEL_NAME = "yolo11n-seg.pt"
EPOCHS = 30
BATCH = 8
IMGSZ = 640

In [ ]:
def mask_to_yolo_multiclass(mask_path, output_txt):

    mask = np.array(Image.open(mask_path))
    classes = np.unique(mask)

    with open(output_txt, "w") as f:
        for cls_id in classes:
            if cls_id == 0:
                continue

            binary_mask = (mask == cls_id).astype(np.uint8)

            contours, _ = cv2.findContours(
                binary_mask,
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE
            )

            for contour in contours:
                contour = contour.squeeze()

                if len(contour.shape) < 2:
                    continue

                h, w = mask.shape
                normalized = contour.astype(float)
                normalized[:, 0] /= w
                normalized[:, 1] /= h

                points = normalized.flatten().tolist()

                if len(points) >= 6:
                    line = f"{cls_id - 1} " + " ".join(map(str, points))
                    f.write(line + "\n")

In [ ]:
def convert_full_dataset():

    if os.path.exists(FULL_DATASET):
        print("Dataset already converted.")
        return

    os.makedirs(f"{FULL_DATASET}/images/train", exist_ok=True)
    os.makedirs(f"{FULL_DATASET}/images/val", exist_ok=True)
    os.makedirs(f"{FULL_DATASET}/labels/train", exist_ok=True)
    os.makedirs(f"{FULL_DATASET}/labels/val", exist_ok=True)

    train_images = sorted(Path(f"{ROOT}/Images/img_dir/train").glob("*.jpg"))
    val_images = sorted(Path(f"{ROOT}/Images/img_dir/test").glob("*.jpg"))

    for split_name, image_list in [("train", train_images), ("val", val_images)]:

        for img_path in image_list:

            img_name = img_path.name
            mask_name = img_name.replace(".jpg", ".png")

            mask_path = f"{ROOT}/Images/ann_dir/{'train' if split_name=='train' else 'test'}/{mask_name}"

            shutil.copy(img_path, f"{FULL_DATASET}/images/{split_name}/{img_name}")

            output_txt = f"{FULL_DATASET}/labels/{split_name}/{img_name.replace('.jpg','.txt')}"

            mask_to_yolo_multiclass(mask_path, output_txt)

convert_full_dataset()

In [ ]:
def create_subset(limit_train, seed):

    subset_path = f"/kaggle/working/subset_{limit_train}_{seed}"

    if os.path.exists(subset_path):
        shutil.rmtree(subset_path)

    os.makedirs(f"{subset_path}/images/train", exist_ok=True)
    os.makedirs(f"{subset_path}/images/val", exist_ok=True)
    os.makedirs(f"{subset_path}/labels/train", exist_ok=True)
    os.makedirs(f"{subset_path}/labels/val", exist_ok=True)

    train_imgs = list(Path(f"{FULL_DATASET}/images/train").glob("*.jpg"))
    val_imgs = list(Path(f"{FULL_DATASET}/images/val").glob("*.jpg"))

    random.seed(seed)
    random.shuffle(train_imgs)
    random.shuffle(val_imgs)

    val_limit = int(limit_train * (len(val_imgs) / len(train_imgs)))

    selected_train = train_imgs[:limit_train]
    selected_val = val_imgs[:val_limit]

    for img_path in selected_train:

        label_path = Path(str(img_path).replace("/images/", "/labels/")).with_suffix(".txt")

        shutil.copy(img_path, f"{subset_path}/images/train/{img_path.name}")
        shutil.copy(label_path, f"{subset_path}/labels/train/{label_path.name}")

    for img_path in selected_val:

        label_path = Path(str(img_path).replace("/images/", "/labels/")).with_suffix(".txt")

        shutil.copy(img_path, f"{subset_path}/images/val/{img_path.name}")
        shutil.copy(label_path, f"{subset_path}/labels/val/{label_path.name}")

    yaml_data = {
        "path": subset_path,
        "train": "images/train",
        "val": "images/val",
        "nc": 103
    }

    yaml_path = f"{subset_path}/dataset.yaml"

    with open(yaml_path, "w") as f:
        yaml.dump(yaml_data, f)

    return yaml_path

In [ ]:
history = []

for sample_size in EXPERIMENTS:

    for run_number in range(RUNS_PER_SIZE):

        seed = random.randint(1, 999999)

        yaml_path = create_subset(sample_size, seed)

        with mlflow.start_run(run_name=f"{sample_size}_run_{run_number+1}"):

            mlflow.log_param("sample_size", sample_size)
            mlflow.log_param("seed", seed)
            mlflow.log_param("epochs", EPOCHS)
            mlflow.log_param("batch", BATCH)
            mlflow.log_param("imgsz", IMGSZ)

            model = YOLO(MODEL_NAME)

            model.train(
                data=yaml_path,
                epochs=EPOCHS,
                imgsz=IMGSZ,
                batch=BATCH,
                seed=seed
            )

            metrics = model.val()

            box_map50 = metrics.box.map50
            seg_map50 = metrics.seg.map50
            precision = metrics.box.mp
            recall = metrics.box.mr

            mlflow.log_metric("box_map50", float(box_map50))
            mlflow.log_metric("seg_map50", float(seg_map50))
            mlflow.log_metric("precision", float(precision))
            mlflow.log_metric("recall", float(recall))

            best_model = "/kaggle/working/runs/segment/train/weights/best.pt"

            if os.path.exists(best_model):
                mlflow.log_artifact(best_model)

            history.append({
                "sample_size": sample_size,
                "run": run_number + 1,
                "seed": seed,
                "box_map50": float(box_map50),
                "seg_map50": float(seg_map50),
                "precision": float(precision),
                "recall": float(recall)
            })

In [ ]:
history_df = pd.DataFrame(history)

history_df.to_csv("/kaggle/working/experiments_history.csv", index=False)

print(history_df)

mlflow.log_artifact("/kaggle/working/experiments_history.csv")